# Первинний аналіз журналу звернень

Ноутбук для розділу 1.2.5 проєкту «Гібридне сховище зображень із прогнозом звернень».
Покладіть цей файл поруч із каталогом `data`, створеним генератором, і виконайте всі ячейки згори вниз.

Потрібні пакети: pandas, numpy, matplotlib, scikit-learn. Усі результати зберігаються в `analysis_output`.
Графіки та висновки обчислюються за TRAIN. TEST використовується лише для перевірки структури й меж вибірки.

Дані є синтетичними. Порівняння прогнозів не замінює експерименту з кешуванням.



In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = str

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
OUTPUT = ROOT / "analysis_output"
FIGURES = OUTPUT / "figures"
assert (DATA_DIR / "metadata.json").is_file(), (
    "Не знайдено data/metadata.json. Відкрийте ноутбук у папці генератора, "
    "де міститься каталог data, або змініть DATA_DIR у цій ячейці."
)
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "figure.dpi": 110,
})
BLUE, TEAL = "#245b87", "#238578"
notes = []

def note(title, text, image_name=None):
    display(Markdown(text))
    notes.append((title, text, image_name))

def save_plot(fig, name):
    fig.savefig(FIGURES / name, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()



## 1 Опис і завантаження набору

Джерело — власний генератор синтетичних успішних user-звернень.
Файл `metadata.json` містить параметри сценарію, версію Python і хеші CSV.
Ідентифікатор зображення використовується для зв'язування записів і не є числовою ознакою моделі.
Усі зображення в цьому сценарії доступні від початку періоду.



In [ ]:
metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
for relative_path, details in metadata["files"].items():
    assert sha256(DATA_DIR / relative_path) == details["sha256"], (
        f"Не збігається SHA-256: {relative_path}. Перевірте набір командою generate_data.py --check."
    )

images = pd.read_csv(DATA_DIR / "raw/images.csv")
logs = pd.read_csv(DATA_DIR / "raw/access_log.csv")
daily = pd.read_csv(DATA_DIR / "processed/daily_features.csv")
logs["timestamp"] = pd.to_datetime(logs["timestamp"], utc=True)
logs["date"] = logs["timestamp"].dt.tz_localize(None).dt.normalize()
daily["date"] = pd.to_datetime(daily["date"])
config = metadata["config"]
FEATURES = ["lag1", "lag2", "lag7", "mean7", "weekday"]
TARGET = "target_count"
RAW_COLUMNS = ["request_id", "timestamp", "image_id", "request_type", "status"]
SPLITS = ["warmup", "train", "validation", "test"]

overview = pd.DataFrame([
    ("Подій", len(logs)), ("Зображень у каталозі", len(images)),
    ("Добових спостережень", len(daily)), ("Період, діб", config["days"]),
    ("Початок", daily["date"].min().date().isoformat()),
    ("Завершення", daily["date"].max().date().isoformat()),
    ("Seed", config["seed"]), ("Часовий пояс", "UTC"),
    ("Версія генератора", metadata["generator_version"]),
    ("Умови поширення в метаданих", metadata.get("data_license", "не зазначено")),
], columns=["Показник", "Значення"])
display(overview)
display(logs[RAW_COLUMNS].head())
if metadata.get("data_license") == "not_selected_by_team":
    note("Умови поширення", "Умови поширення набору ще не обрані командою. Перед завершенням опису даних визначте їх і зафіксуйте у відповідному файлі. Наявні метадані не підтверджують поширення набору під CC0.")



## 2 Перевірка структури та історичних ознак

Перевіряємо повноту сітки «зображення — доба» та заново обчислюємо ціль і лаги з журналу.
Справжні нульові доби відрізняються від невідомої історії на початку періоду.
Повторні запити до одного зображення в ту саму секунду дозволені, якщо request_id різні.



In [ ]:
assert len(logs) == config["n_requests"]
assert len(images) == config["n_images"]
assert images["image_id"].is_unique and logs["request_id"].is_unique
assert logs[RAW_COLUMNS].notna().all().all()
assert logs["image_id"].isin(images["image_id"]).all()
assert logs["timestamp"].is_monotonic_increasing
assert not daily.duplicated(["image_id", "date"]).any()

start = pd.Timestamp(config["start_date"])
dates = pd.date_range(start, periods=config["days"], freq="D")
assert logs["date"].between(dates.min(), dates.max()).all()
successful = logs.loc[(logs["request_type"] == "user") & (logs["status"] == "success")].copy()
grid = pd.MultiIndex.from_product(
    [sorted(images["image_id"].tolist()), dates], names=["image_id", "date"]
)
rebuilt = successful.groupby(["image_id", "date"]).size().reindex(grid, fill_value=0).to_frame(TARGET)
history = rebuilt[TARGET].groupby(level="image_id", sort=False)
for lag in (1, 2, 7):
    rebuilt[f"lag{lag}"] = history.shift(lag)
rebuilt["mean7"] = history.transform(lambda s: s.shift(1).rolling(7, min_periods=7).mean())
rebuilt["weekday"] = rebuilt.index.get_level_values("date").weekday
provided = daily.set_index(["image_id", "date"]).sort_index()
assert provided.index.equals(rebuilt.index)
for column in [TARGET] + FEATURES:
    assert np.allclose(provided[column], rebuilt[column], equal_nan=True, atol=1e-8, rtol=0), column
assert int(daily[TARGET].sum()) == len(successful)

assert daily["split"].isin(SPLITS).all()
for name in SPLITS:
    section = daily.loc[daily["split"] == name]
    bounds = metadata["splits"][name]
    assert len(section) == bounds["n_rows"]
    assert section["date"].min() == pd.Timestamp(bounds["start_date"])
    assert section["date"].max() == pd.Timestamp(bounds["end_date"])
    assert section["date"].nunique() == bounds["n_dates"]
    assert section["date"].between(bounds["start_date"], bounds["end_date"]).all()

split_summary = daily.groupby("split").agg(
    rows=("image_id", "size"), dates=("date", "nunique"),
    date_start=("date", "min"), date_end=("date", "max"),
).reindex(SPLITS)
display(split_summary)
train = daily.loc[daily["split"] == "train"].copy()
validation = daily.loc[daily["split"] == "validation"].copy()
warmup = daily.loc[daily["split"] == "warmup"].copy()
assert train[FEATURES + [TARGET]].notna().all().all()
assert validation[FEATURES + [TARGET]].notna().all().all()
train_logs = successful.loc[successful["date"].between(train["date"].min(), train["date"].max())].copy()
display(train[FEATURES + [TARGET]].describe().round(3))
note("Перевірка даних", f"Структурні перевірки пройдено. У журналі {len(logs):,} подій, у каталозі {len(images):,} зображень. Цільові значення та всі історичні ознаки відповідають вихідному журналу. Межі дат є спільними для всіх зображень.")



## 3 Розподіл цільової змінної

Це задача регресії. Замість балансу класів досліджуємо добовий попит, зокрема частку нульових спостережень.



In [ ]:
zero_share = float(train[TARGET].eq(0).mean())
fig, ax = plt.subplots(figsize=(8.8, 4.4))
ax.hist(np.log1p(train[TARGET]), bins=45, color=BLUE, edgecolor="white", linewidth=0.5)
ax.set(title="Розподіл добового попиту — TRAIN", xlabel="ln(1 + кількість звернень за добу)", ylabel="Кількість спостережень")
ax.grid(axis="y", alpha=0.2)
save_plot(fig, "01_target_distribution.png")
note("Розподіл попиту", f"Нульовий попит мають **{zero_share:.1%}** навчальних пар «зображення — доба». Ці спостереження зберігаються й ураховуються в MAE. Для показу всього діапазону на графіку використано ln(1 + count); ціль у таблиці залишається в початкових одиницях.", "01_target_distribution.png")



## 4 Популярність за рангом

Ранжуємо зображення лише за TRAIN. Ранг на графіку не передається моделі як ознака.



In [ ]:
popularity = train.groupby("image_id")[TARGET].sum().reindex(images["image_id"], fill_value=0).sort_values(ascending=False, kind="stable")
ranks = np.arange(1, len(popularity) + 1)
positive = popularity.to_numpy() > 0
top10_share = float(popularity.head(10).sum() / popularity.sum())
fig, ax = plt.subplots(figsize=(8.8, 4.4))
ax.loglog(ranks[positive], popularity.to_numpy()[positive], color=BLUE, linewidth=1.8)
ax.set(title="Популярність зображень за рангом — TRAIN", xlabel="Ранг популярності (логарифмічна шкала)", ylabel="Кількість звернень (логарифмічна шкала)")
ax.grid(alpha=0.2)
save_plot(fig, "02_popularity_rank.png")
note("Нерівномірність популярності", f"На 10 найпопулярніших зображень припадає **{top10_share:.1%}** звернень TRAIN. Зображень без звернень у цій частині: **{int((popularity == 0).sum())}**; вони не показані на логарифмічній осі, але залишаються в даних. Генератор використовує базові ваги Ціпфа із сезонністю та шумом; цей графік сам по собі не доводить точного закону розподілу.", "02_popularity_rank.png")



## 5 Часові ряди зображень

Вибираємо представників високого, середнього й нижчого попиту за TRAIN. Вертикальні шкали панелей різні.



In [ ]:
positions = sorted(set([0, len(popularity) // 2, 3 * len(popularity) // 4]))
selected_ids = [int(popularity.index[p]) for p in positions]
fig, axes = plt.subplots(len(selected_ids), 1, figsize=(9.2, 7.2), sharex=True, squeeze=False)
for ax, image_id in zip(axes[:, 0], selected_ids):
    series = train.loc[train["image_id"] == image_id].sort_values("date")
    ax.plot(series["date"], series[TARGET], color=BLUE, linewidth=1.4, marker=".", markersize=3)
    ax.set(title=f"Зображення {image_id}", ylabel="Звернень")
    ax.grid(alpha=0.2)
axes[-1, 0].set_xlabel("Дата, UTC")
fig.suptitle("Добові часові ряди — TRAIN", y=1.01)
fig.autofmt_xdate()
fig.tight_layout()
save_plot(fig, "03_time_series.png")
note("Часова структура", f"Для огляду за TRAIN вибрано зображення **{', '.join(map(str, selected_ids))}**. Ряди показують фактичні добові коливання в згенерованому сценарії. Для прогнозу використовуватимемо лаги та календарну ознаку; часові спостереження не перемішуються випадково.", "03_time_series.png")



## 6 Добовий профіль

Кількість подій кожної години ділимо на число повних навчальних діб.



In [ ]:
hourly = train_logs.groupby(train_logs["timestamp"].dt.hour).size().reindex(range(24), fill_value=0) / train["date"].nunique()
peak_hour = int(hourly.idxmax())
fig, ax = plt.subplots(figsize=(9.2, 4.4))
ax.bar(hourly.index, hourly.values, color=BLUE)
ax.set(title="Середня кількість звернень за годинами — TRAIN", xlabel="Година, UTC", ylabel="Звернень на годину за добу")
ax.set_xticks(range(0, 24, 2))
ax.grid(axis="y", alpha=0.2)
save_plot(fig, "04_hourly_profile.png")
note("Добова сезонність", f"Найбільше середнє навантаження зафіксовано о **{peak_hour:02d}:00 UTC**: {hourly.max():.1f} звернення на годину за добу. Годинний порядок потрібний для відтворення навантаження на кеш; поточна модель прогнозує добовий підсумок.", "04_hourly_profile.png")



## 7 Тижневий профіль

Порівнюємо середні добові підсумки, а не сумарні значення за нерівні періоди.



In [ ]:
day_labels = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Нд"]
daily_total = train.groupby("date")[TARGET].sum()
weekly = daily_total.groupby(daily_total.index.weekday).mean().reindex(range(7))
peak_weekday = int(weekly.idxmax())
fig, ax = plt.subplots(figsize=(8.8, 4.4))
ax.bar(day_labels, weekly.values, color=TEAL)
ax.set(title="Середній добовий попит за днями тижня — TRAIN", xlabel="День тижня", ylabel="Звернень за добу")
ax.grid(axis="y", alpha=0.2)
save_plot(fig, "05_weekly_profile.png")
note("Тижнева сезонність", f"Найбільший середній добовий попит у TRAIN має день **{day_labels[peak_weekday]}**: {weekly.max():.1f} звернення. У моделі залишаємо weekday та lag7. Наявність календарного профілю не гарантує переваги моделі над базовим прогнозом — це перевірятиметься окремо.", "05_weekly_profile.png")



## 8 Пропуски

Окремо показуємо вихідний журнал і невідому історію перших семи діб. Штучно додавати помилки заради аналізу не потрібно.



In [ ]:
raw_missing = logs[RAW_COLUMNS].isna().sum()
lag_columns = ["lag1", "lag2", "lag7", "mean7"]
missing_warmup = warmup[lag_columns].isna().mean() * 100
missing_train = train[lag_columns].isna().mean() * 100
fig, axes = plt.subplots(2, 1, figsize=(8.8, 7.4))
axes[0].bar(raw_missing.index, raw_missing.values, color=BLUE)
axes[0].set(title="Пропуски у вихідному журналі", ylabel="Кількість")
axes[0].set_ylim(0, max(1, float(raw_missing.max()) * 1.2))
axes[0].tick_params(axis="x", rotation=15)
for i, count in enumerate(raw_missing):
    axes[0].text(i, float(count) + 0.02, str(int(count)), ha="center", va="bottom")
x = np.arange(len(lag_columns))
axes[1].bar(x - 0.18, missing_warmup, width=0.36, color=BLUE, label="Warmup")
axes[1].bar(x + 0.18, missing_train, width=0.36, color=TEAL, label="Train")
axes[1].set(title="Пропуски історичних ознак", ylabel="Частка, %", xticks=x, xticklabels=lag_columns, ylim=(0, 110))
axes[1].legend(frameon=False)
fig.tight_layout()
save_plot(fig, "06_missing_values.png")
note("Обробка пропусків", f"У перевірених полях вихідного журналу виявлено **{int(raw_missing.sum())}** пропусків. Пропуски лагів у warmup є наслідком відсутньої попередньої історії. Після виключення warmup у навчальних ознаках залишається **{int(train[FEATURES].isna().sum().sum())}** пропусків. Перші сім діб використовуємо як історію, їхні неповні рядки не навчають модель.", "06_missing_values.png")



## 9 Кореляції

Обчислюємо кореляцію Спірмена за TRAIN. image_id є ключем, а weekday — календарною категорією; їх не включаємо до цієї матриці.



In [ ]:
corr_columns = ["lag1", "lag2", "lag7", "mean7", TARGET]
corr = train[corr_columns].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7.0, 5.7))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_columns)), corr_columns, rotation=30, ha="right")
ax.set_yticks(range(len(corr_columns)), corr_columns)
ax.set_title("Кореляція Спірмена — TRAIN")
for i in range(len(corr_columns)):
    for j in range(len(corr_columns)):
        value = float(corr.iloc[i, j])
        ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="white" if abs(value) > 0.65 else "black")
fig.colorbar(im, ax=ax, shrink=0.85, label="Коефіцієнт кореляції")
fig.tight_layout()
save_plot(fig, "07_correlation_matrix.png")
target_corr = corr[TARGET].drop(TARGET)
strongest_feature = target_corr.abs().idxmax()
note("Зв'язок ознак із ціллю", f"Найбільший за модулем зв'язок із target_count серед показаних ознак має **{strongest_feature}**: коефіцієнт Спірмена **{target_corr[strongest_feature]:.3f}**. Кореляція описує зв'язок, але не доводить причинності або відсутності витоку. Відсутність підглядання перевірено окремим перерахунком лагів із минулих діб.", "07_correlation_matrix.png")



## 10 Екстремальні значення

Значення вище Q3 + 1.5 IQR позначаємо як статистично екстремальні, але не видаляємо автоматично.
Для нерівномірного попиту високі значення можуть бути змістовними спостереженнями.



In [ ]:
q1, q3 = train[TARGET].quantile([0.25, 0.75])
upper_bound = float(q3 + 1.5 * (q3 - q1))
extreme_count = int(train[TARGET].gt(upper_bound).sum())
groups = [train.loc[train["weekday"] == i, TARGET].to_numpy() for i in range(7)]
fig, ax = plt.subplots(figsize=(9.2, 4.8))
boxes = ax.boxplot(groups, tick_labels=day_labels, patch_artist=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.2},
    medianprops={"color": "#ad3f23", "linewidth": 1.4})
for box in boxes["boxes"]:
    box.set_facecolor("#dce9f1")
ax.set_yscale("symlog", linthresh=1)
ax.set(title="Розкид добового попиту за днями тижня — TRAIN", xlabel="День тижня", ylabel="Звернень за добу (шкала symlog)")
ax.grid(axis="y", alpha=0.2)
save_plot(fig, "08_outliers_boxplot.png")
note("Викиди та піки", f"Межа Q3 + 1.5 IQR дорівнює **{upper_bound:.2f}**. Вище неї розташовано **{extreme_count:,}** спостережень ({extreme_count / len(train):.1%} TRAIN). Високі значення з коректними ідентифікаторами та часом залишаємо: для цього сценарію вони є частиною попиту. На графіку використано symlog, щоб показати нулі й великі значення одночасно.", "08_outliers_boxplot.png")



## 11 Метрика і базовий прогноз

MAE — середня абсолютна різниця між фактичним і прогнозованим добовим попитом, в одиницях «звернень на зображення за добу».
Базовий прогноз «буде як учора» дорівнює lag1. Він не потребує навчання; його не слід називати навченою моделлю.
Зараз оцінюємо VALIDATION. TEST буде оцінено після фіксації параметрів і вибору між Random Forest та LightGBM.



In [ ]:
baseline_mae = float(mean_absolute_error(validation[TARGET], validation["lag1"]))
baseline = pd.DataFrame([{
    "strategy": "yesterday", "split": "validation", "rows": len(validation),
    "date_start": validation["date"].min().date().isoformat(),
    "date_end": validation["date"].max().date().isoformat(),
    "mae": baseline_mae,
}])
display(baseline)
note("Метрика та валідація", f"MAE базового прогнозу «буде як учора» на VALIDATION: **{baseline_mae:.6f}** за {len(validation):,} парами «зображення — доба», включно з нульовим попитом. Random Forest і LightGBM порівнюватимуться на тих самих датах та ознаках. Параметри й алгоритм вибираються за VALIDATION; TEST не використано для підбору або оцінювання у цьому ноутбуці.")
note("Обмеження", "Результати характеризують заданий синтетичний сценарій. Зменшення MAE саме по собі не доводить поліпшення кешування. Hit rate, затримку отримання та трафік prefetch потрібно виміряти окремо за однакових потоків і місткості кешу. Під час майбутнього тестування прогнозуємо послідовно по одній добі, використовуючи лише завершену історію.")



## 12 Експорт матеріалів

У Word перенесіть зображення з analysis_output/figures та перевірені висновки з findings.md.
Ці висновки обчислені за поточними CSV, їх потрібно прочитати й узгодити з формулюваннями вашого звіту.
Збережіть виконаний ноутбук у репозиторії разом із кодом генератора та зафіксованим набором даних.



In [ ]:
summary = {
    "source": metadata.get("source"), "data_license": metadata.get("data_license"),
    "generator_version": metadata.get("generator_version"), "config": config,
    "dataset_files": metadata["files"], "checks": "OK",
    "counts": {"events": len(logs), "catalog_images": len(images), "daily_rows": len(daily)},
    "split_bounds": metadata["splits"],
    "train": {"rows": len(train), "zero_target_share": zero_share,
        "top10_request_share": top10_share, "iqr_upper_bound": upper_bound,
        "extreme_rows": extreme_count, "peak_hour_utc": peak_hour,
        "peak_weekday": peak_weekday, "strongest_spearman_feature": strongest_feature,
        "strongest_spearman_value": float(target_corr[strongest_feature])},
    "baseline_validation": {"mae": baseline_mae, "rows": len(validation)},
    "test_performance_evaluated": False,
    "environment": {"python": platform.python_version(), **{
        package: importlib.metadata.version(package)
        for package in ["numpy", "pandas", "matplotlib", "scikit-learn"]}},
}
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
baseline.to_csv(OUTPUT / "baseline_validation.csv", index=False)
split_summary.to_csv(OUTPUT / "split_summary.csv")
corr.to_csv(OUTPUT / "correlations_train.csv")
parts = ["# Висновки первинного аналізу\n", "Обчислено за поточними CSV; основний аналіз виконано за TRAIN.\n"]
for title, text, image_name in notes:
    parts.extend([f"## {title}\n", text + "\n"])
    if image_name:
        parts.append(f"Графік: figures/{image_name}\n")
(OUTPUT / "findings.md").write_text("\n".join(parts), encoding="utf-8")
print("Готово. Графіки та висновки:", OUTPUT.resolve())
print("PNG:", len(list(FIGURES.glob("*.png"))))
